# 03a - Auto-Tuning with MetaSchedule: Getting Started

In notebook 02, we manually explored schedule primitives — reorder, tile, unroll, vectorize, parallel.  
Each gave a speedup, but we only tried a handful of parameter choices. The real optimization space is enormous:

- How many levels to tile? Which factors at each level?
- Which loops to parallelize, vectorize, or unroll?
- What order should the loops be in?
- What combination of all of the above works best *together*?

For a 1024×1024 matmul on CPU, the number of valid schedule combinations easily reaches **millions**.  
**MetaSchedule** is TVM's framework for searching this space automatically.

This notebook covers:
1. Measuring the **untuned baseline** (raw, no schedule transforms)
2. Running an auto-tuning session with **`tune_tir()`**
3. Inspecting and understanding the results
4. Benchmarking **tuned vs untuned** code
5. Understanding the **tuning database** and resumption behavior


Related material: [MLC Course Chapter 4](https://book.mlc.ai/chapter_auto_program_optimization/index.html)

In [1]:
import os
import json
import shutil

import numpy as np
import tvm
import tvm_ffi
from tvm import meta_schedule as ms

from workloads import get_workload, get_workload_flops

## Baseline: untuned 1024×1024 matmul

We switch from 256×256 (notebooks 01–02) to **1024×1024** — large enough for auto-tuning to make a dramatic difference.  
First, let's measure what the raw, unscheduled TIR produces.

In [6]:
mod = get_workload("matmul_1024")
target = tvm.target.Target("llvm --num-cores=8")
FLOPS = get_workload_flops("matmul_1024")
N = 1024

# Build with no schedule transformations
built_untuned = tvm.build(mod, target=target)

# Allocate buffers
np.random.seed(42)
A_np = np.random.randn(N, N).astype("float32")
B_np = np.random.randn(N, N).astype("float32")

A_tvm = tvm_ffi.from_dlpack(A_np)
B_tvm = tvm_ffi.from_dlpack(B_np)

dev = tvm.cpu(0)
timer = built_untuned.time_evaluator("main", dev, repeat=3, number=5)
C_np = np.empty((N, N), dtype="float32")
C_tvm = tvm_ffi.from_dlpack(C_np)
result_untuned = timer(A_tvm, B_tvm, C_tvm)

print(f"Untuned: {result_untuned.mean*1e3:.2f} ms  "
      f"({FLOPS / result_untuned.mean / 1e9:.2f} GFLOPS)")
print()
print("Original TIR (triple nested loop, no optimizations):")
mod.show()

Untuned: 3154.08 ms  (0.68 GFLOPS)

Original TIR (triple nested loop, no optimizations):


## What is MetaSchedule?

MetaSchedule is TVM's auto-tuning framework. Given an unscheduled TIR program and a hardware target,
it **automatically searches** for the best combination of schedule primitives (the same ones from notebook 02:
split, reorder, parallel, vectorize, unroll, etc.).

At a high level, the tuning loop works like this:

```
                        ┌─────────────────────────┐
     IRModule ─────────►│  Generate Search Space   │  Schedule rules define what
     + Target           │  (schedule rules)        │  transforms are legal
                        └────────────┬────────────┘
                                     │ design space (schedule templates)
                                     ▼
                        ┌─────────────────────────┐
           ┌───────────►│  Search Strategy         │  Pick promising candidates
           │            │  (evolutionary search)   │  using cost model + mutations
           │            └────────────┬────────────┘
           │                         │ batch of candidate schedules
           │                         ▼
           │            ┌─────────────────────────┐
           │            │  Builder                 │  Compile each candidate
           │            └────────────┬────────────┘
           │                         │ compiled artifacts
           │                         ▼
           │            ┌─────────────────────────┐
           │            │  Runner                  │  Measure wall-clock time
           │            └────────────┬────────────┘
           │                         │ measured latencies
           │                         ▼
           │            ┌─────────────────────────┐
           │            │  Database                │  Store results, update
           └────────────┤  + Cost Model            │  cost model predictions
              feedback  └─────────────────────────┘
```

Each iteration:
1. The **search strategy** proposes a batch of candidate schedules
2. The **builder** compiles them
3. The **runner** measures their actual runtime on the hardware
4. Results are stored in the **database** and used to update the **cost model**
5. Repeat until the trial budget is exhausted

We will dive into each component in notebook 03b. For now, let's use the high-level API.

## ⚠️  Important: the `work_dir` and re-run behavior

Before running our first tuning session, there is one critical thing to understand:

> **MetaSchedule always runs `max_trials_global` NEW trials**, regardless of what's already in `work_dir`.  
> Previous results are loaded and used to **guide the search** (better initial population for the  
> evolutionary algorithm), but they do **not** count toward the trial budget.

This means:
- **First run** (empty work_dir): runs `max_trials_global` trials, saves results to database.
- **Second run** (same work_dir): runs **another** `max_trials_global` trials (guided by previous data), appends to database.
- After two runs with `max_trials_global=64`, the database contains **~128 records**, not 64.

This is important to understand because:
- **Re-running a tuning cell is not free** — it does the full work again, every time
- Each re-run benefits from previous data (the evolutionary search starts from better candidates)
- **To start fresh**: delete the `work_dir` or use a different path

In the cell below, we explicitly clean the work directory before tuning.

## Running `tune_tir()`

The `tune_tir()` function is the simplest entry point. It takes:
- **`mod`** — the IRModule containing our PrimFunc
- **`target`** — the hardware target (e.g. `"llvm"` for CPU)
- **`work_dir`** — where to store the database and logs
- **`max_trials_global`** — total number of candidate schedules to try
- **`num_trials_per_iter`** — batch size per iteration

With `max_trials_global=64` and `num_trials_per_iter=16`, tuning will run **4 iterations** of 16 candidates each.  
This is deliberately small to keep the tutorial fast — real tuning often uses 1000+ trials.

In [7]:
WORK_DIR = "./tune_matmul_1024_basics"

# Clean previous run to ensure we start fresh
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

db = ms.tune_tir(
    mod=mod,
    target=target,
    work_dir=WORK_DIR,
    max_trials_global=64,
    num_trials_per_iter=16,
    seed=42,
)

2026-03-10 17:27:43 [INFO] Logging directory: ./tune_matmul_1024_basics/logs
2026-03-10 17:27:43 [INFO] LocalBuilder: max_workers = 20
2026-03-10 17:27:45 [INFO] LocalRunner: max_workers = 1
2026-03-10 17:27:47 [INFO] [task_scheduler.cc:168] Initializing Task #0: "main"


,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,main,2147483648,1,N/A,N/A,N/A,0,



Total trials: 0
Total latency (us): 0

2026-03-10 17:27:47 [DEBUG] [task_scheduler.cc:327] 
 ID | Name |       FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
---------------------------------------------------------------------------------------------------------
  0 | main | 2147483648 |      1 |            N/A |          N/A |                   N/A |      0 |      
---------------------------------------------------------------------------------------------------------
Total trials: 0
Total latency (us): 0

2026-03-10 17:27:47 [INFO] [task_scheduler.cc:189] TaskScheduler picks Task #0: "main"
2026-03-10 17:27:47 [INFO] [task_scheduler.cc:202] Sending 16 sample(s) to builder
2026-03-10 17:27:58 [INFO] [task_scheduler.cc:204] Sending 16 sample(s) to runner
2026-03-10 17:28:06 [DEBUG] XGB iter   0: tr-p-rmse: 0.557848	tr-a-peak@32: 0.871896	tr-rmse: 0.275191	tr-rmse: 0.275191
2026-03-10 17:28:06 [DEBUG] XGB iter  25: tr-p-rmse: 0.048125	tr-a-peak

,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,main,2147483648,1,227.7716,9428.2309,9428.2309,16,


2026-03-10 17:28:06 [DEBUG] [task_scheduler.cc:327] 
 ID | Name |       FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
---------------------------------------------------------------------------------------------------------
  0 | main | 2147483648 |      1 |       227.7716 |    9428.2309 |             9428.2309 |     16 |      
---------------------------------------------------------------------------------------------------------
Total trials: 16
Total latency (us): 9428.23


Total trials: 16
Total latency (us): 9428.23

2026-03-10 17:28:06 [INFO] [task_scheduler.cc:189] TaskScheduler picks Task #0: "main"
2026-03-10 17:28:06 [INFO] [task_scheduler.cc:202] Sending 16 sample(s) to builder
2026-03-10 17:28:20 [INFO] [task_scheduler.cc:204] Sending 16 sample(s) to runner
2026-03-10 17:28:40 [DEBUG] XGB validation: p-rmse: 0.277668	a-peak@32: 0.954547
2026-03-10 17:28:41 [DEBUG] XGB iter   0: tr-p-rmse: 0.527119	tr-a-peak@32: 0.917240	tr-rmse: 0.2

...

2026-03-10 17:29:05 [DEBUG] [task_scheduler.cc:327] 
 ID | Name |       FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
---------------------------------------------------------------------------------------------------------
  0 | main | 2147483648 |      1 |       227.7716 |    9428.2309 |             9428.2309 |     48 |      
---------------------------------------------------------------------------------------------------------
Total trials: 48
Total latency (us): 9428.23


Total trials: 48
Total latency (us): 9428.23

2026-03-10 17:29:05 [INFO] [task_scheduler.cc:189] TaskScheduler picks Task #0: "main"
2026-03-10 17:29:05 [INFO] [task_scheduler.cc:202] Sending 16 sample(s) to builder
2026-03-10 17:29:16 [INFO] [task_scheduler.cc:204] Sending 16 sample(s) to runner
2026-03-10 17:29:21 [DEBUG] XGB validation: p-rmse: 0.245451	a-peak@32: 0.868183
2026-03-10 17:29:21 [DEBUG] XGB iter   0: tr-p-rmse: 0.485681	tr-a-peak@32: 0.807025	tr-rmse: 0.2

,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,main,2147483648,1,260.9787,8228.5776,8228.5776,64,



Total trials: 64
Total latency (us): 8228.58

2026-03-10 17:29:21 [DEBUG] [task_scheduler.cc:327] 
 ID | Name |       FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
---------------------------------------------------------------------------------------------------------
  0 | main | 2147483648 |      1 |       260.9787 |    8228.5776 |             8228.5776 |     64 |      
---------------------------------------------------------------------------------------------------------
Total trials: 64
Total latency (us): 8228.58

2026-03-10 17:29:21 [INFO] [task_scheduler.cc:269] Task #0 has finished. Remaining task(s): 0


,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,main,2147483648,1,260.9787,8228.5776,8228.5776,64,Y


2026-03-10 17:29:21 [DEBUG] [task_scheduler.cc:327] 
 ID | Name |       FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
---------------------------------------------------------------------------------------------------------
  0 | main | 2147483648 |      1 |       260.9787 |    8228.5776 |             8228.5776 |     64 |    Y 
---------------------------------------------------------------------------------------------------------
Total trials: 64
Total latency (us): 8228.58


Total trials: 64
Total latency (us): 8228.58



### Reading the tuning output

While tuning runs, you'll see output like:

```
[Task #0: "main"]  Generating design space...
[Task #0: "main"]  Sending 16 sample(s) to builder
[Task #0: "main"]  Sending 16 sample(s) to runner
[Task #0: "main"]  [Updated]  Trial #16: ... 12.3456 ms.  Best: 5.6789 ms  ...
```

Key things to watch:
- **"Generating design space"**: MetaSchedule creates schedule templates using rules (covered in 03b)
- **"Sending N sample(s) to builder/runner"**: a batch of candidates is being compiled and measured
- **"Best: X ms"**: the best latency found so far — this should decrease over iterations

## Inspecting the best schedule

`tune_tir()` returns a `Database` object. We can query it for the best schedule found.

In [9]:
# Retrieve the best schedule from the database
sch = db.query_schedule(mod, target, "main")

print("Schedule trace (the sequence of primitives MetaSchedule applied):")
print("=" * 70)
sch.trace.show()
print("=" * 70)

Schedule trace (the sequence of primitives MetaSchedule applied):


### Reading the trace

The trace above is a sequence of **the same primitives from notebook 02**: `get_block`, `get_loops`,
`split`, `reorder`, `parallel`, `vectorize`, `unroll`, etc.

The difference is that MetaSchedule chose the **parameters automatically** — tile sizes, loop order,
which loops to annotate — by measuring real performance on your hardware.

Let's see what the resulting TIR looks like compared to the original.

In [10]:
print("=== ORIGINAL TIR (untuned) ===")
mod.show()

print("\n=== TUNED TIR (best schedule) ===")
sch.mod.show()

=== ORIGINAL TIR (untuned) ===



=== TUNED TIR (best schedule) ===


## Benchmarking: tuned vs untuned

The real test — compile the tuned schedule and compare wall-clock time.

In [11]:
# Build the tuned module
built_tuned = tvm.build(sch.mod, target=target)

# Benchmark
timer_tuned = built_tuned.time_evaluator("main", dev, repeat=3, number=5)
C_tuned_np = np.empty((N, N), dtype="float32")
C_tuned_tvm = tvm_ffi.from_dlpack(C_tuned_np)
result_tuned = timer_tuned(A_tvm, B_tvm, C_tuned_tvm)

print(f"{'Untuned:':<10} {result_untuned.mean*1e3:8.2f} ms  "
      f"({FLOPS / result_untuned.mean / 1e9:6.2f} GFLOPS)")
print(f"{'Tuned:':<10} {result_tuned.mean*1e3:8.2f} ms  "
      f"({FLOPS / result_tuned.mean / 1e9:6.2f} GFLOPS)")
print(f"\nSpeedup: {result_untuned.mean / result_tuned.mean:.1f}x")

Untuned:    3154.08 ms  (  0.68 GFLOPS)
Tuned:         8.95 ms  (239.89 GFLOPS)

Speedup: 352.3x


[17:31:22] /home/arthur/Documents/Projects/fromsource/d2l-tvm/tvm23/src/runtime/threading_backend.cc:348: Warning: more than two frequencies detected! Forced big_count_ to 16


In [12]:
# Verify correctness
C_ref = A_np @ B_np

C_untuned_np = np.empty((N, N), dtype="float32")
C_untuned_tvm = tvm_ffi.from_dlpack(C_untuned_np)
built_untuned(A_tvm, B_tvm, C_untuned_tvm)

C_tuned_np2 = np.empty((N, N), dtype="float32")
C_tuned_tvm2 = tvm_ffi.from_dlpack(C_tuned_np2)
built_tuned(A_tvm, B_tvm, C_tuned_tvm2)

print("Untuned matches NumPy:", np.allclose(C_untuned_np, C_ref, rtol=1e-3, atol=1e-3))
print("Tuned   matches NumPy:", np.allclose(C_tuned_np2, C_ref, rtol=1e-3, atol=1e-3))

Untuned matches NumPy: True
Tuned   matches NumPy: True


## The tuning database on disk

MetaSchedule stores everything in `work_dir`. Let's look at what's there.

In [13]:
# List files in the work directory
for root, dirs, files in os.walk(WORK_DIR):
    level = root.replace(WORK_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in sorted(files):
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f"{indent}  {f}  ({size:,} bytes)")

tune_matmul_1024_basics/
  database_tuning_record.json  (126,125 bytes)
  database_workload.json  (28,168 bytes)
  logs/
    tvm.meta_schedule.logging.task_0_main.log  (29,611 bytes)
    tvm.meta_schedule.logging.task_scheduler.log  (7,164 bytes)


In [ ]:
# The tuning records are stored as one JSON object per line.
# Each line is: [workload_index, [trace, run_secs, target, args_info]]
record_path = os.path.join(WORK_DIR, "database_tuning_record.json")
with open(record_path) as f:
    lines = f.readlines()

print(f"Total tuning records: {len(lines)}")
print(f"\nFirst record structure:")
record = json.loads(lines[0])

workload_idx = record[0]         # index into database_workload.json
tuning_data = record[1]          # [trace, run_secs, target, args_info]
run_secs = tuning_data[1]        # list of measured runtimes

print(f"  Workload index: {workload_idx}")
print(f"  Measured runtimes: {run_secs}")
print(f"  Mean: {np.mean(run_secs)*1e3:.2f} ms")

## Demonstrating re-run behavior

Let's see what happens when we call `tune_tir()` again with the **same work directory**.

As explained above, MetaSchedule will run **another 64 trials** — the existing database
is loaded to guide the evolutionary search, but the trial counter resets to zero.
New records are **appended** to the database file.

In [ ]:
# Check record count BEFORE re-run
with open(record_path) as f:
    records_before = len(f.readlines())
print(f"Records before re-run: {records_before}")

# Same work_dir, same budget → runs 64 NEW trials (not a no-op!)
db_rerun = ms.tune_tir(
    mod=mod,
    target=target,
    work_dir=WORK_DIR,
    max_trials_global=64,
    num_trials_per_iter=16,
    seed=42,
)

# Check record count AFTER re-run
with open(record_path) as f:
    records_after = len(f.readlines())
print(f"\nRecords after re-run:  {records_after}")
print(f"New records added:     {records_after - records_before}")
print(f"\nThe database grew — re-running is NOT free, it always does full work.")

In [ ]:
# The best schedule may have improved thanks to the extra data
sch_after_rerun = db_rerun.query_schedule(mod, target, "main")
built_after_rerun = tvm.build(sch_after_rerun.mod, target=target)

C_tmp = tvm_ffi.from_dlpack(np.empty((N, N), dtype="float32"))
result_after_rerun = built_after_rerun.time_evaluator(
    "main", dev, repeat=3, number=5
)(A_tvm, B_tvm, C_tmp)

print(f"{'Best after 64 trials:':<28} {result_tuned.mean*1e3:8.2f} ms")
print(f"{'Best after 128 trials:':<28} {result_after_rerun.mean*1e3:8.2f} ms")
print(f"\nMore data can improve results, but each re-run costs time."
      f"\nTo avoid accidental re-runs, clean work_dir only when intentional.")

## Summary

| Step | API | Key insight |
|---|---|---|
| Tune | `ms.tune_tir(mod, target, work_dir, max_trials)` | One call to search the schedule space |
| Query best | `db.query_schedule(mod, target, "main")` | Returns a `tvm.tir.Schedule` with concrete decisions |
| Inspect | `sch.trace` / `sch.mod.show()` | See exactly which primitives + parameters were chosen |
| Build | `tvm.build(sch.mod, target)` | Same compilation path as manual schedules |
| Benchmark | `built.time_evaluator(...)` | Verify the speedup is real |

Key takeaways:
- MetaSchedule applies the **same primitives** from notebook 02, but chooses parameters automatically
- Results persist in `work_dir` — re-running **always runs the full trial budget again** (not a resume/skip)
- Previous results **guide** the evolutionary search but don't reduce the budget
- To start fresh: delete `work_dir`. Re-running without deleting accumulates more records
- Even with just 64 trials, auto-tuning typically finds a schedule **much faster** than the untuned baseline

**Next notebook (03b):** What happens inside MetaSchedule — schedule rules, search space generation,
search strategies, and how to customize the tuning process.